# 🐾 Pet Face Classification
### Internship Project 2 — Deep Learning | CNN-Based Image Classification

**Dataset:** [The Oxford-IIIT Pet Dataset](https://www.kaggle.com/datasets/tanlikesmath/the-oxfordiiit-pet-dataset)  
**Task:** Multi-class classification of pet (cat & dog) face images using a Convolutional Neural Network (CNN) with Transfer Learning.

---

## Step 1: Install & Import Required Libraries

In [ ]:
# Install necessary packages (run once)
!pip install tensorflow matplotlib seaborn scikit-learn opencv-python kaggle

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Sklearn
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Step 2: Download Dataset from Kaggle

> **How to set up Kaggle API:**
> 1. Go to [kaggle.com](https://www.kaggle.com) → Your Account → API → Create New Token
> 2. A `kaggle.json` file will be downloaded
> 3. Place it at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\Users\<user>\.kaggle\kaggle.json` (Windows)

In [ ]:
# Setup Kaggle credentials
import os

# If running on Google Colab, upload kaggle.json manually:
# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download the Oxford-IIIT Pet Dataset
!kaggle datasets download -d tanlikesmath/the-oxfordiiit-pet-dataset
!unzip -q the-oxfordiiit-pet-dataset.zip -d pet_dataset
print("Dataset downloaded and extracted successfully!")

## Step 3: Explore & Understand the Dataset

In [ ]:
# Define dataset path
DATASET_PATH = 'pet_dataset/images'

# List all image files
all_images = [f for f in os.listdir(DATASET_PATH) if f.endswith('.jpg')]
print(f"Total images found: {len(all_images)}")
print("Sample files:", all_images[:5])

In [ ]:
# Extract class names from filenames
# Filename format: BreedName_index.jpg (e.g., Abyssinian_1.jpg)
# Uppercase = Cat, Lowercase = Dog

def extract_class(filename):
    """Extract breed class from filename."""
    name = '_'.join(filename.split('_')[:-1])  # Remove trailing index
    return name

def get_species(filename):
    """Cats have Capitalized breed names, Dogs have lowercase."""
    return 'Cat' if filename[0].isupper() else 'Dog'

# Build a DataFrame
df = pd.DataFrame({'filename': all_images})
df['filepath'] = df['filename'].apply(lambda x: os.path.join(DATASET_PATH, x))
df['breed'] = df['filename'].apply(extract_class)
df['species'] = df['filename'].apply(get_species)

print(df.head(10))
print(f"\nTotal Breeds: {df['breed'].nunique()}")
print(f"Species Distribution:\n{df['species'].value_counts()}")

In [ ]:
# Visualize class distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
df['species'].value_counts().plot(kind='bar', color=['#FF6B6B', '#4ECDC4'], edgecolor='black')
plt.title('Cat vs Dog Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Species')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
top_breeds = df['breed'].value_counts().head(20)
top_breeds.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 20 Breeds', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
sample_df = df.groupby('breed').first().reset_index().head(15)

for idx, ax in enumerate(axes.flatten()):
    if idx < len(sample_df):
        img = mpimg.imread(sample_df.iloc[idx]['filepath'])
        ax.imshow(img)
        ax.set_title(f"{sample_df.iloc[idx]['breed']}\n({sample_df.iloc[idx]['species']})",
                     fontsize=8, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Pet Images from Dataset', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

## Step 4: Data Preprocessing & Splitting

In [ ]:
from sklearn.model_selection import train_test_split

# We'll classify by SPECIES (Binary: Cat vs Dog) — can be changed to breed (37 classes)
# To do breed classification, change label_col = 'breed'
label_col = 'species'   # Change to 'breed' for 37-class classification

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Train / Validation / Test split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[label_col], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df[label_col], random_state=42)

print(f"Training samples   : {len(train_df)}")
print(f"Validation samples : {len(val_df)}")
print(f"Test samples       : {len(test_df)}")
print(f"\nClasses: {df[label_col].unique()}")

In [ ]:
# Image Data Generators with Augmentation for Training
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Create generators
train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col=label_col,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col=label_col,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col=label_col,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_gen.class_indices.keys())
num_classes = len(class_names)
print(f"\nClass indices: {train_gen.class_indices}")

In [ ]:
# Visualize Augmented Images
sample_batch, sample_labels = next(train_gen)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, ax in enumerate(axes.flatten()):
    img = sample_batch[i]
    # Reverse MobileNetV2 preprocessing for display
    img_display = (img + 1) / 2.0  # [-1,1] → [0,1]
    img_display = np.clip(img_display, 0, 1)
    label_idx = np.argmax(sample_labels[i])
    ax.imshow(img_display)
    ax.set_title(class_names[label_idx], fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Augmented Training Images', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('augmented_images.png', dpi=150)
plt.show()

## Step 5: Build the CNN Model (Transfer Learning — MobileNetV2)

In [ ]:
# Load pretrained MobileNetV2 (ImageNet weights, without top layers)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model layers initially
base_model.trainable = False

# Build full model
inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Model compiled successfully!")
print(f"Total parameters     : {model.count_params():,}")
print(f"Trainable parameters : {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

## Step 6: Train the Model — Phase 1 (Feature Extraction)

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_pet_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

EPOCHS_PHASE1 = 15

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 1 Training Complete!")

## Step 7: Fine-Tuning — Phase 2 (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 50 layers of base model for fine-tuning
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 50

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total base layers     : {len(base_model.layers)}")
print(f"Unfrozen from layer   : {fine_tune_at}")
print(f"Trainable layers now  : {sum(1 for l in base_model.layers if l.trainable)}")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_PHASE2 = 10

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 2 Fine-Tuning Complete!")

## Step 8: Training History Visualization

In [ ]:
# Combine history from both phases
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

history = combine_histories(history1, history2)
epochs_range = range(1, len(history['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(epochs_range, history['accuracy'], 'b-o', label='Train Accuracy', linewidth=2)
axes[0].plot(epochs_range, history['val_accuracy'], 'r-o', label='Val Accuracy', linewidth=2)
axes[0].axvline(EPOCHS_PHASE1, color='gray', linestyle='--', alpha=0.7, label='Fine-tune Start')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_range, history['loss'], 'b-o', label='Train Loss', linewidth=2)
axes[1].plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
axes[1].axvline(EPOCHS_PHASE1, color='gray', linestyle='--', alpha=0.7, label='Fine-tune Start')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History (Phase 1 + Phase 2 Fine-Tuning)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## Step 9: Model Evaluation on Test Set

In [ ]:
# Load best saved model
best_model = keras.models.load_model('best_pet_model.h5')

# Evaluate on test set
test_loss, test_accuracy = best_model.evaluate(test_gen, verbose=1)
print(f"\n📊 Test Loss     : {test_loss:.4f}")
print(f"📊 Test Accuracy : {test_accuracy*100:.2f}%")

In [ ]:
# Predictions
test_gen.reset()
y_pred_probs = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

# Classification Report
print("\n📋 Classification Report:")
print("="*60)
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(max(6, num_classes), max(5, num_classes - 1)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix — Pet Face Classification', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Step 10: Visualize Predictions

In [ ]:
# Show sample predictions with confidence scores
test_gen.reset()
sample_batch_x, sample_batch_y = next(test_gen)
preds = best_model.predict(sample_batch_x)

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for i, ax in enumerate(axes.flatten()[:15]):
    img = sample_batch_x[i]
    img_display = (img + 1) / 2.0
    img_display = np.clip(img_display, 0, 1)

    true_label = class_names[np.argmax(sample_batch_y[i])]
    pred_label = class_names[np.argmax(preds[i])]
    confidence = np.max(preds[i]) * 100

    color = 'green' if true_label == pred_label else 'red'
    ax.imshow(img_display)
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Model Predictions (Green = Correct, Red = Wrong)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150)
plt.show()

## Step 11: Predict on a Custom Image

In [ ]:
def predict_pet(image_path, model, class_names, img_size=(224, 224)):
    """
    Predict pet species/breed from a single image.
    
    Args:
        image_path (str): Path to the image file
        model: Trained Keras model
        class_names (list): List of class labels
        img_size (tuple): Target image size
    """
    # Load and preprocess
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, img_size)
    img_array = np.expand_dims(img_resized.astype('float32'), axis=0)
    img_preprocessed = preprocess_input(img_array)

    # Predict
    predictions = model.predict(img_preprocessed, verbose=0)[0]
    top_idx = np.argsort(predictions)[::-1][:3]

    # Display
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Input Image", fontsize=13, fontweight='bold')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    top_labels = [class_names[i] for i in top_idx]
    top_scores = [predictions[i] * 100 for i in top_idx]
    colors = ['#2ecc71', '#3498db', '#e74c3c']
    bars = plt.barh(top_labels[::-1], top_scores[::-1], color=colors[::-1], edgecolor='black')
    plt.xlabel('Confidence (%)')
    plt.title('Top-3 Predictions', fontsize=13, fontweight='bold')
    plt.xlim(0, 100)
    for bar, score in zip(bars, top_scores[::-1]):
        plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{score:.1f}%', va='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig('custom_prediction.png', dpi=150)
    plt.show()

    print(f"\n🏆 Predicted: {class_names[top_idx[0]]} ({predictions[top_idx[0]]*100:.1f}% confidence)")
    return class_names[top_idx[0]], predictions[top_idx[0]]


# ---- Usage Example ----
# Replace with any local pet image path
sample_img_path = test_df.iloc[0]['filepath']
predicted_class, confidence = predict_pet(sample_img_path, best_model, class_names)

## Step 12: Save the Final Model

In [ ]:
# Save in Keras native format
best_model.save('pet_face_classifier_final.keras')
print("✅ Model saved as: pet_face_classifier_final.keras")

# Also save as TFLite for mobile deployment
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()
with open('pet_face_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print("✅ TFLite model saved as: pet_face_classifier.tflite")

# Save class names
import json
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print("✅ Class names saved as: class_names.json")

## Step 13: Summary & Results

In [ ]:
print("="*60)
print("       PET FACE CLASSIFICATION — PROJECT SUMMARY")
print("="*60)
print(f"  Dataset       : The Oxford-IIIT Pet Dataset (Kaggle)")
print(f"  Total Images  : {len(df)}")
print(f"  Task          : {label_col.capitalize()} Classification ({num_classes} classes)")
print(f"  Model         : MobileNetV2 (Transfer Learning + Fine-Tuning)")
print(f"  Image Size    : {IMG_SIZE}")
print(f"  Batch Size    : {BATCH_SIZE}")
print(f"  Train Split   : {len(train_df)} images")
print(f"  Val Split     : {len(val_df)} images")
print(f"  Test Split    : {len(test_df)} images")
print("-"*60)
print(f"  Test Accuracy : {test_accuracy*100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")
print("="*60)
print("✅ Project Complete!")

---
## 📌 References
- **Dataset:** [The Oxford-IIIT Pet Dataset on Kaggle](https://www.kaggle.com/datasets/tanlikesmath/the-oxfordiiit-pet-dataset)
- **Original Paper:** Parkhi et al., *Cats and Dogs*, CVPR 2012
- **Model:** [MobileNetV2](https://keras.io/api/applications/mobilenet/#mobilenetv2-function) — pretrained on ImageNet
- **Framework:** TensorFlow 2.x / Keras

---
*Internship Project 2 — Classification of Pet's Faces*